加载模型。
这里有几个细节：
- load_model_and_processor() 使用官方自定义模型类。
- 默认采用 BF16。
- 默认采用 flash_attention_2。
- 默认 model.eval()。
- 如果模型权重已经在 Hugging Face 缓存中，会优先使用缓存。

In [6]:
import sys
import torch

from pathlib import Path
from PIL import Image
from IPython.display import display

PROJECT_ROOT = Path(
    "/home/20261024_1/Workspace/qyh/VLM-test"
)

# 将 CV-Bench 评测目录加入 Python 搜索路径
ADAPTER_DIR = (
    PROJECT_ROOT
    / "evaluation"
    / "cv-bench"
    / "xiaomi_lens"
)

sys.path.insert(0, str(ADAPTER_DIR))

# 直接复用之前全量评测使用的 Adapter
from xiaomi_lens_adapter import XiaomiLensAdapter

# 加载模型
adapter = XiaomiLensAdapter(
    model_path="SeerRay-Lab/Xiaomi-Lens-0",
    device="cuda:0",
    attn_implementation="flash_attention_2",
)

load_info = adapter.load()

print("模型加载成功！")
print(load_info)

Loading weights: 100%|██████████| 749/749 [00:02<00:00, 252.96it/s]


模型加载成功！
{'model_name': 'Xiaomi-Lens-0', 'model_path': '/home/20261024_1/.cache/huggingface/hub/models--SeerRay-Lab--Xiaomi-Lens-0/snapshots/68f57cdde1a19f751e78b18af04efa8ca8bd7fce', 'attn_implementation': 'flash_attention_2', 'dtype': 'bfloat16', 'load_seconds': 4.997434735996649, 'vram_after_load': {'allocated_gib': 17.42439365386963, 'reserved_gib': 17.44140625, 'peak_allocated_gib': 17.42440366744995, 'peak_reserved_gib': 17.44140625, 'gpu_used_gib': 25.13360595703125, 'gpu_total_gib': 94.9600830078125}, 'eos_token_ids': {248044, 248046}, 'think_close_token_id': 248069, 'mask_token_id': 248077}


In [8]:
def test_xiaomi(image, question, mode="off", max_new_tokens=2048):

    # 1. 构建输入
    prepared = adapter.build_inputs(
        image=image,
        instruction=question,
        mode=mode,
    )

    # 2. 验证 Thinking 模式是否真正生效
    print("Thinking mode:", mode)
    print("enable_thinking:", prepared["enable_thinking"])
    print("Prompt verified:", prepared["mode_verified"])

    if not prepared["mode_verified"]:
        raise RuntimeError("Thinking 模式验证失败！")

    # 3. 执行推理
    result = adapter.generate(
        prepared=prepared,
        mode=mode,
        max_new_tokens=max_new_tokens,
    )

    # 4. 分离 Thinking 与最终回答
    raw_text = result["raw_generation"]

    if mode == "on":
        _, separator, answer = raw_text.partition("</think>")

        if not separator:
            answer = "[未检测到 </think>，可能尚未完成思考]"
    else:
        answer = raw_text

    # 5. 输出结果
    print("\n" + "=" * 60)
    print("最终回答")
    print("=" * 60)
    print(answer.strip())

    print("\n" + "=" * 60)
    print("性能统计")
    print("=" * 60)

    print("生成 Token:", result["generated_tokens"])
    print("Thinking Token:", result["thinking_tokens"])
    print("Answer Token:", result["answer_tokens"])
    print("生成耗时:", f"{result['generate_seconds']:.3f}s")
    print("生成速度:", f"{result['tokens_per_second']:.2f} tokens/s")
    print("Peak VRAM:", f"{result['vram']['peak_allocated_gib']:.2f} GiB")
    print("Finish reason:", result["finish_reason"])
    return result

### 测试1

1. 尝试中文+非思考模式。模型能够正确回答。

In [9]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，
定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 2 列的那本书，并回答：
1. 这本书的书脊是什么主要颜色？
2. 如果书脊上有可辨认的文字，请准确抄写书名。
3. 简要描述这本书的位置。
请注意：
必须根据图片实际内容判断，不要猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
根据图片内容，第 2 行、第 2 列的书是《Music》。

1. 这本书的书脊主要颜色是**白色**（或米白色）。
2. 书脊上的文字是：**Music**。
3. 这本书位于书架的**第二行、第二列**，在《Space》的右侧和《Animals》的左侧。<|im_end|>

性能统计
生成 Token: 82
Thinking Token: 0
Answer Token: 82
生成耗时: 2.267s
生成速度: 36.18 tokens/s
Peak VRAM: 17.79 GiB
Finish reason: eos


{'raw_generation': '根据图片内容，第 2 行、第 2 列的书是《Music》。\n\n1. 这本书的书脊主要颜色是**白色**（或米白色）。\n2. 书脊上的文字是：**Music**。\n3. 这本书位于书架的**第二行、第二列**，在《Space》的右侧和《Animals》的左侧。<|im_end|>',
 'generated_tokens': 82,
 'thinking_tokens': 0,
 'answer_tokens': 82,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 2.2666295950766653,
 'tokens_per_second': 36.17706226818523,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 17.459603309631348,
  'reserved_gib': 17.85546875,
  'peak_allocated_gib': 17.790788650512695,
  'peak_reserved_gib': 17.85546875,
  'gpu_used_gib': 23.16552734375,
  'gpu_total_gib': 94.9600830078125}}

尝试中文+非思考模式+直接回答指令。模型能正确回答。

In [10]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 2 列的那本书，并回答：
1.这本书的书脊是什么主要颜色？
2.如果书脊上有可辨认的文字，请准确抄写书名。
3.简要描述这本书的位置，说明你找到的是哪一行、哪一列。
请注意：
必须根据图片实际内容判断，不要根据书架的常见排列规律猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
请直接输出答案，而不要输出你的分析过程。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
1. 这本书的书脊主要颜色是白色。
2. 书名是“Music”。
3. 这本书位于第 2 行、第 2 列。<|im_end|>

性能统计
生成 Token: 39
Thinking Token: 0
Answer Token: 39
生成耗时: 1.134s
生成速度: 34.38 tokens/s
Peak VRAM: 17.80 GiB
Finish reason: eos


{'raw_generation': '1. 这本书的书脊主要颜色是白色。\n2. 书名是“Music”。\n3. 这本书位于第 2 行、第 2 列。<|im_end|>',
 'generated_tokens': 39,
 'thinking_tokens': 0,
 'answer_tokens': 39,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 1.134426181903109,
 'tokens_per_second': 34.37861415942794,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 17.459604263305664,
  'reserved_gib': 17.85546875,
  'peak_allocated_gib': 17.797080516815186,
  'peak_reserved_gib': 17.85546875,
  'gpu_used_gib': 23.16552734375,
  'gpu_total_gib': 94.9600830078125}}

### 测试5

1. 中文+非思考模式。翻车了。

In [12]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试5.jpg")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从左到右的顺序排列，最左边的黄色的书作为第1本。现在请找到第14本书，并回答：
1.这本书的书脊是什么主要颜色？
2.如果书脊上有可辨认的文字，请准确抄写书名。
3.简要描述这本书的位置，说明你找到的是哪一行、哪一列。
请注意：
必须根据图片实际内容判断，不要根据书架的常见排列规律猜测。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
根据图片内容，从左到右依次排列书籍：
1. 黄色书脊《苏东坡传》
2. 浅蓝色书脊《数学Ⅱ(女孩)》
3. 浅粉色书脊《数学Ⅱ(女孩)》
4. 浅紫色书脊《数学Ⅱ(女孩)》
5. 红色书脊《现代汉语词典》
6. 绿色书脊《图解电的基础知识入门》
7. 绿色书脊《图解电机基础知识入门》
8. 绿色书脊《图解机械基础知识入门》
9. 白色书脊《电磁场基本原理66课》
10. 白色书脊《电工电路基本原理66课》
11. 白色书脊《电子电路基本原理66课》
12. 白色书脊《苏东坡新传》（上）
13. 白色书脊《苏东坡新传》（下）
14. 蓝色书脊《新英汉小词典》

第14本书是《新英汉小词典》，书脊主要颜色为蓝色。

1. 这本书的书脊主要颜色是蓝色。
2. 书名：新英汉小词典
3. 位置：第1行，从左数第14本。<|im_end|>

性能统计
生成 Token: 280
Thinking Token: 0
Answer Token: 280
生成耗时: 8.164s
生成速度: 34.30 tokens/s
Peak VRAM: 19.02 GiB
Finish reason: eos


{'raw_generation': '根据图片内容，从左到右依次排列书籍：\n1. 黄色书脊《苏东坡传》\n2. 浅蓝色书脊《数学Ⅱ(女孩)》\n3. 浅粉色书脊《数学Ⅱ(女孩)》\n4. 浅紫色书脊《数学Ⅱ(女孩)》\n5. 红色书脊《现代汉语词典》\n6. 绿色书脊《图解电的基础知识入门》\n7. 绿色书脊《图解电机基础知识入门》\n8. 绿色书脊《图解机械基础知识入门》\n9. 白色书脊《电磁场基本原理66课》\n10. 白色书脊《电工电路基本原理66课》\n11. 白色书脊《电子电路基本原理66课》\n12. 白色书脊《苏东坡新传》（上）\n13. 白色书脊《苏东坡新传》（下）\n14. 蓝色书脊《新英汉小词典》\n\n第14本书是《新英汉小词典》，书脊主要颜色为蓝色。\n\n1. 这本书的书脊主要颜色是蓝色。\n2. 书名：新英汉小词典\n3. 位置：第1行，从左数第14本。<|im_end|>',
 'generated_tokens': 280,
 'thinking_tokens': 0,
 'answer_tokens': 280,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 8.163595960941166,
 'tokens_per_second': 34.298610727388244,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 17.61031198501587,
  'reserved_gib': 19.970703125,
  'peak_allocated_gib': 19.02257490158081,
  'peak_reserved_gib': 19.970703125,
  'gpu_used_gib': 25.28076171875,
  'gpu_total_gib': 94.9600830078125}}

2. 下面尝试中文+思考模式。未翻车。

In [13]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试5.jpg")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从左到右的顺序排列，最左边的黄色的书作为第1本。现在请找到第14本书，并回答：
1.这本书的书脊是什么主要颜色？
2.如果书脊上有可辨认的文字，请准确抄写书名。
3.简要描述这本书的位置，说明你找到的是哪一行、哪一列。
请注意：
必须根据图片实际内容判断，不要根据书架的常见排列规律猜测。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="on",
)

Thinking mode: on
enable_thinking: True
Prompt verified: True

最终回答
1. 这本书的书脊主要颜色是白色。
2. 书脊上的书名是：苏东坡新传。
3. 这本书位于第一行，从左往右数第14本。<|im_end|>

性能统计
生成 Token: 505
Thinking Token: 458
Answer Token: 46
生成耗时: 14.237s
生成速度: 35.47 tokens/s
Peak VRAM: 19.02 GiB
Finish reason: eos


{'raw_generation': '1.  **确定目标位置**：\n    *   第一行书籍从左到右依次编号：\n        *   第1本：黄色书脊《苏东坡传》\n        *   第2本：浅蓝色书脊《数学Ⅱ(女孩)》\n        *   第3本：粉色书脊《数学Ⅱ(女孩)》\n        *   第4本：浅蓝色书脊《数学Ⅱ(女孩)》\n        *   第5本：粉色书脊《数学Ⅱ(女孩)》\n        *   第6本：红色书脊《现代汉语词典》\n        *   第7本：白色书脊《图解电的基础知识入门》\n        *   第8本：白色书脊《图解电机基础知识入门》\n        *   第9本：白色书脊《图解机械基础知识入门》\n        *   第10本：白色书脊《电磁场基本原理66课》\n        *   第11本：白色书脊《电工电路基本原理66课》\n        *   第12本：白色书脊《电子电路基本原理66课》\n        *   第13本：白色书脊《苏东坡新传》（上）\n        *   第14本：白色书脊《苏东坡新传》（下）\n    *   目标书籍是第14本，即《苏东坡新传》（下）。\n\n2.  **分析目标书籍特征**：\n    *   **书脊颜色**：书脊主体为白色，顶部和底部有棕色/米色区域。\n    *   **书名**：书脊上的文字清晰可见，为“苏东坡新传”。\n    *   **位置描述**：位于第一行，从左往右数第14本。\n\n3.  **构建回答**：\n    *   书脊主要颜色：白色。\n    *   书名：苏东坡新传。\n    *   位置：第一行，从左往右数第14本。\n</think>\n\n1. 这本书的书脊主要颜色是白色。\n2. 书脊上的书名是：苏东坡新传。\n3. 这本书位于第一行，从左往右数第14本。<|im_end|>',
 'generated_tokens': 505,
 'thinking_tokens': 458,
 'answer_tokens': 46,
 'closed_thinking': True,
 'finish_reason': 'eos',
 'truncated': False,
 

3. 尝试英文+非思考模式。没有翻车，但是有一定的列举分析。

In [14]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试5.jpg")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
Please carefully observe the bookshelf in the image.
We arrange the books in the bookshelf in order from left to right, with the yellow book on the far left as the 1st book. Now please find the 14th book and answer:
What is the main color of this book's spine?
If there is legible text on the spine, please accurately copy the book title.
Briefly describe the position of this book, explaining which row and which column you found it in.
Please note:
You must judge based on the actual content of the image, and do not guess based on common arrangement patterns of bookshelves.
If the text cannot be recognized, clearly answer "cannot be recognized" and do not fabricate a book title.
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
Based on the image provided, here is the analysis of the bookshelf from left to right:

1.  **1st book:** Yellow spine, "Su Dongpo" (苏东坡).
2.  **2nd book:** Light blue spine, "Math II (Girls x Girls)" (数学Ⅱ(女×孩)).
3.  **3rd book:** Pink spine, "Math II (Girls x Girls) 3" (数学Ⅱ(女×孩)(3)).
4.  **4th book:** Light purple spine, "Math II (Girls x Girls) 2" (数学Ⅱ(女×孩)(2)).
5.  **5th book:** Light purple spine, "Math II (Girls x Girls) 4" (数学Ⅱ(女×孩)(4)).
6.  **6th book:** Red spine, "Modern Chinese Dictionary" (现代汉语词典).
7.  **7th book:** Green spine, "Illustrated Electric Basics Knowledge Introduction" (图解电的基础知识入门).
8.  **8th book:** Green spine, "Illustrated Electric Machine Basics Knowledge Introduction" (图解电机基础知识入门).
9.  **9th book:** Green spine, "Illustrated Mechanical Basics Knowledge Introduction" (图解机械基础知识入门).
10. **10th book:** White spine, "Electromagnetic Basic Principles 66 Lessons" (电磁场基本原理66课).
11. **11th book:** 

{'raw_generation': 'Based on the image provided, here is the analysis of the bookshelf from left to right:\n\n1.  **1st book:** Yellow spine, "Su Dongpo" (苏东坡).\n2.  **2nd book:** Light blue spine, "Math II (Girls x Girls)" (数学Ⅱ(女×孩)).\n3.  **3rd book:** Pink spine, "Math II (Girls x Girls) 3" (数学Ⅱ(女×孩)(3)).\n4.  **4th book:** Light purple spine, "Math II (Girls x Girls) 2" (数学Ⅱ(女×孩)(2)).\n5.  **5th book:** Light purple spine, "Math II (Girls x Girls) 4" (数学Ⅱ(女×孩)(4)).\n6.  **6th book:** Red spine, "Modern Chinese Dictionary" (现代汉语词典).\n7.  **7th book:** Green spine, "Illustrated Electric Basics Knowledge Introduction" (图解电的基础知识入门).\n8.  **8th book:** Green spine, "Illustrated Electric Machine Basics Knowledge Introduction" (图解电机基础知识入门).\n9.  **9th book:** Green spine, "Illustrated Mechanical Basics Knowledge Introduction" (图解机械基础知识入门).\n10. **10th book:** White spine, "Electromagnetic Basic Principles 66 Lessons" (电磁场基本原理66课).\n11. **11th book:** White spine, "Electrical Circuit Basic

4. 下面尝试英文+非思考模式+直接输入指令。翻大车了。

In [15]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试5.jpg")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
Please carefully observe the bookshelf in the image.
We arrange the books in the bookshelf in order from left to right, with the yellow book on the far left as the 1st book. Now please find the 14th book and answer:
What is the main color of this book's spine?
If there is legible text on the spine, please accurately copy the book title.
Briefly describe the position of this book, explaining which row and which column you found it in.
Please note:
You must judge based on the actual content of the image, and do not guess based on common arrangement patterns of bookshelves.
If the text cannot be recognized, clearly answer "cannot be recognized" and do not fabricate a book title.
Please output the answer directly, and do not output your analysis process.
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
The 14th book is the one with the white spine and black text reading "电子电路基本原理66课". Its main color is white. It is located in the second row, fourth column.<|im_end|>

性能统计
生成 Token: 43
Thinking Token: 0
Answer Token: 43
生成耗时: 1.814s
生成速度: 23.71 tokens/s
Peak VRAM: 19.03 GiB
Finish reason: eos


{'raw_generation': 'The 14th book is the one with the white spine and black text reading "电子电路基本原理66课". Its main color is white. It is located in the second row, fourth column.<|im_end|>',
 'generated_tokens': 43,
 'thinking_tokens': 0,
 'answer_tokens': 43,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 1.8135161318350583,
 'tokens_per_second': 23.71084505131433,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 17.61031198501587,
  'reserved_gib': 19.970703125,
  'peak_allocated_gib': 19.02941131591797,
  'peak_reserved_gib': 19.970703125,
  'gpu_used_gib': 25.28076171875,
  'gpu_total_gib': 94.9600830078125}}